## Exercise

- Create a notebook which does following:

    - Insert at least **3 different coins** into the `coins` table.

    - Insert **5 daily price records** for each coin into the `prices` table.

    - Create a business question (what is the average price for each coin? what is the highest price for each coin?)
    
    - Write an SQL query to solve the business question

In [1]:
import sqlite3
import os
import pandas as pd

In [2]:
db_path = "cryptocurrency.db"

if os.path.exists(db_path):
    os.remove(db_path)
    print("Database file deleted.")
else:
    print("Database file does not exist.")

Database file deleted.


In [3]:
# Create or connect to a database file
conn = sqlite3.connect(db_path)

# Create a cursor object to execute SQL commands
cursor = conn.cursor()

# Step 2: Create tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS coins (
    coin_id INTEGER PRIMARY KEY,
    name TEXT,
    symbol TEXT
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS prices (
    price_id INTEGER PRIMARY KEY,
    coin_id INTEGER,
    price REAL,
    date TEXT,
FOREIGN KEY (coin_id) REFERENCES coins (coin_id)
)
""")
conn.commit() # Commit table creation

In [4]:
# Insert at least 3 different coins into the `coins` table.

coins_data = [
    (1, 'Solana', 'SOL'), ## List of Tuples
    (2, 'Chainlink', 'LINK'),   
    (3, 'Hyperliquid', 'HYPE'),
    (4, 'Avalanche', 'AVAX'),
    (5, 'Litecoin', 'LTC'),
    (6, 'Uniswap', 'UNI'),
    (7, 'Toncoin', 'TON'),
    (8, 'Mantle', 'MNT') # Will have no price
]

cursor.executemany("INSERT OR IGNORE INTO coins VALUES (?, ?, ?)", coins_data)

# Insert 5 daily price records for each coin into the `prices` table.

prices_data = [
    (1, 1, 117.83, '2025-08-11'),
    (2, 2, 22.91, '2025-08-11'),
    (3, 3, 43.36, '2025-08-11'),
    (4, 4, 23.69, '2025-08-11'),
    (5, 5, 121.16, '2025-08-11'),
    (6, 6, 11.11, '2025-08-11'),
    (7, 7, 3.40, '2025-08-11'),
    (8, 8, 1.02, '2025-08-11')
]
cursor.executemany("INSERT OR IGNORE INTO prices VALUES (?, ?, ?, ?)", prices_data)

conn.commit() # Commit data inserts 


In [5]:
# Get all coins
cursor.execute("SELECT * FROM coins")
all_coins = cursor.fetchall() # Get all results as a list of tuples
print("All coins:", all_coins)

# Get all prices
cursor.execute("SELECT * FROM prices")
all_prices = cursor.fetchall() # Get all results as a list of tuples
print("All prices:", all_prices)

All coins: [(1, 'Solana', 'SOL'), (2, 'Chainlink', 'LINK'), (3, 'Hyperliquid', 'HYPE'), (4, 'Avalanche', 'AVAX'), (5, 'Litecoin', 'LTC'), (6, 'Uniswap', 'UNI'), (7, 'Toncoin', 'TON'), (8, 'Mantle', 'MNT')]
All prices: [(1, 1, 117.83, '2025-08-11'), (2, 2, 22.91, '2025-08-11'), (3, 3, 43.36, '2025-08-11'), (4, 4, 23.69, '2025-08-11'), (5, 5, 121.16, '2025-08-11'), (6, 6, 11.11, '2025-08-11'), (7, 7, 3.4, '2025-08-11'), (8, 8, 1.02, '2025-08-11')]


In [9]:
# Run LEFT JOIN query
with sqlite3.connect(db_path) as conn:
    query = """
    SELECT c.name, c.symbol, p.price, p.date
    FROM coins c
    LEFT JOIN prices p ON c.coin_id = p.coin_id
    """
    df = pd.read_sql(query, conn)

print("\nAll coins with prices (if available):")
print(df)


All coins with prices (if available):
          name symbol   price        date
0       Solana    SOL  117.83  2025-08-11
1    Chainlink   LINK   22.91  2025-08-11
2  Hyperliquid   HYPE   43.36  2025-08-11
3    Avalanche   AVAX   23.69  2025-08-11
4     Litecoin    LTC  121.16  2025-08-11
5      Uniswap    UNI   11.11  2025-08-11
6      Toncoin    TON    3.40  2025-08-11
7       Mantle    MNT    1.02  2025-08-11


In [11]:
df = pd.read_sql(query, conn)

print("\nBasic statistics:")
print(df.describe())

print("\nAverage price:", df['price'].mean())

# Filtering coins less than $100
cheap = df[df['price'] < 100]
print("\nCheap coins:")
print(cheap[['name', 'price']])



Basic statistics:
            price
count    8.000000
mean    43.060000
std     49.037234
min      1.020000
25%      9.182500
50%     23.300000
75%     61.977500
max    121.160000

Average price: 43.06

Cheap coins:
          name  price
1    Chainlink  22.91
2  Hyperliquid  43.36
3    Avalanche  23.69
5      Uniswap  11.11
6      Toncoin   3.40
7       Mantle   1.02


In [12]:
# Close connection
conn.close()